# Silver Layer - Data Transformation

This notebook performs data quality checks, normalization, validation, and joins all tables from the bronze layer into a single analytical table.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configuration
bronze_catalog = "workspace"
bronze_schema = "bronze"
silver_catalog = "workspace"
silver_schema = "silver"

# Create silver schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")
print(f"Schema {silver_catalog}.{silver_schema} is ready.")

Schema workspace.silver is ready.


In [0]:
# Load all tables from bronze layer
print("Loading data from bronze layer...\n")

fact_sales = spark.table(f"{bronze_catalog}.{bronze_schema}.factsalestable")
dim_customer = spark.table(f"{bronze_catalog}.{bronze_schema}.dimcustomertable")
dim_date = spark.table(f"{bronze_catalog}.{bronze_schema}.dimdatetable")
dim_region = spark.table(f"{bronze_catalog}.{bronze_schema}.dimregiontable")
dim_product = spark.table(f"{bronze_catalog}.{bronze_schema}.dimproducttable")
dim_product_subcategory = spark.table(f"{bronze_catalog}.{bronze_schema}.dimproductsubcategorytable")
dim_product_category = spark.table(f"{bronze_catalog}.{bronze_schema}.dimproductcategorytable")

print(f"✓ Fact Sales: {fact_sales.count():,} rows")
print(f"✓ Dim Customer: {dim_customer.count():,} rows")
print(f"✓ Dim Date: {dim_date.count():,} rows")
print(f"✓ Dim Region: {dim_region.count():,} rows")
print(f"✓ Dim Product: {dim_product.count():,} rows")
print(f"✓ Dim Product Subcategory: {dim_product_subcategory.count():,} rows")
print(f"✓ Dim Product Category: {dim_product_category.count():,} rows")

Loading data from bronze layer...

✓ Fact Sales: 114,390 rows
✓ Dim Customer: 9,999 rows
✓ Dim Date: 1,188 rows
✓ Dim Region: 655 rows
✓ Dim Product: 400 rows
✓ Dim Product Subcategory: 40 rows
✓ Dim Product Category: 5 rows


In [0]:
print("\n" + "="*80)
print("STEP 2: CHECKING FOR DUPLICATES")
print("="*80 + "\n")

# Check duplicates in fact table
fact_total = fact_sales.count()
fact_distinct = fact_sales.select("ProductKey", "OrderDateKey", "CustomerKey", "OrderNumber").distinct().count()
fact_duplicates = fact_total - fact_distinct
print(f"Fact Sales - Total: {fact_total:,}, Distinct: {fact_distinct:,}, Duplicates: {fact_duplicates:,}")

# Check duplicates in dimension tables
tables_to_check = [
    ("Dim Customer", dim_customer, "CustomerKey"),
    ("Dim Date", dim_date, "DateKey"),
    ("Dim Region", dim_region, "GeographyKey"),
    ("Dim Product", dim_product, "ProductKey"),
    ("Dim Product Subcategory", dim_product_subcategory, "ProductSubcategoryKey"),
    ("Dim Product Category", dim_product_category, "ProductCategoryKey")
]

for table_name, df, key_col in tables_to_check:
    total = df.count()
    distinct = df.select(key_col).distinct().count()
    duplicates = total - distinct
    print(f"{table_name} - Total: {total:,}, Distinct: {distinct:,}, Duplicates: {duplicates:,}")

# Remove duplicates from fact table if any
if fact_duplicates > 0:
    fact_sales_clean = fact_sales.dropDuplicates(["ProductKey", "OrderDateKey", "CustomerKey", "OrderNumber"])
    print(f"\n✓ Removed {fact_duplicates:,} duplicate rows from Fact Sales")
else:
    fact_sales_clean = fact_sales
    print("\n✓ No duplicates found in Fact Sales")


STEP 2: CHECKING FOR DUPLICATES

Fact Sales - Total: 114,390, Distinct: 57,188, Duplicates: 57,202
Dim Customer - Total: 9,999, Distinct: 9,999, Duplicates: 0
Dim Date - Total: 1,188, Distinct: 1,188, Duplicates: 0
Dim Region - Total: 655, Distinct: 655, Duplicates: 0
Dim Product - Total: 400, Distinct: 400, Duplicates: 0
Dim Product Subcategory - Total: 40, Distinct: 40, Duplicates: 0
Dim Product Category - Total: 5, Distinct: 5, Duplicates: 0

✓ Removed 57,202 duplicate rows from Fact Sales


In [0]:
print("\n" + "="*80)
print("STEP 3: CHECKING FOR NULL VALUES")
print("="*80 + "\n")

def check_nulls(df, table_name):
    print(f"\n{table_name}:")
    total_rows = df.count()
    null_counts = []
    
    for col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        if null_count > 0:
            null_counts.append((col, null_count, null_pct))
    
    if null_counts:
        for col, count, pct in null_counts:
            print(f"  - {col}: {count:,} nulls ({pct:.2f}%)")
    else:
        print("  ✓ No null values found")
    
    return null_counts

# Check nulls in all tables
fact_nulls = check_nulls(fact_sales_clean, "Fact Sales")
cust_nulls = check_nulls(dim_customer, "Dim Customer")
date_nulls = check_nulls(dim_date, "Dim Date")
region_nulls = check_nulls(dim_region, "Dim Region")
product_nulls = check_nulls(dim_product, "Dim Product")
subcat_nulls = check_nulls(dim_product_subcategory, "Dim Product Subcategory")
cat_nulls = check_nulls(dim_product_category, "Dim Product Category")


STEP 3: CHECKING FOR NULL VALUES


Fact Sales:
  - Gender: 57,188 nulls (100.00%)

Dim Customer:
  ✓ No null values found

Dim Date:
  ✓ No null values found

Dim Region:
  ✓ No null values found

Dim Product:
  - Color: 56 nulls (14.00%)

Dim Product Subcategory:
  ✓ No null values found

Dim Product Category:
  ✓ No null values found


In [0]:
print("\n" + "="*80)
print("STEP 4: HANDLING NULLS AND STANDARDIZATION")
print("="*80 + "\n")

# Handle nulls in fact table - Gender column has nulls, fill with 'Unknown'
fact_sales_normalized = fact_sales_clean.withColumn(
    "Gender",
    F.when(F.col("Gender").isNull(), "Unknown").otherwise(F.col("Gender"))
)

print("✓ Filled null Gender values with 'Unknown' in Fact Sales")

# Standardize text columns - trim whitespace and convert to proper case
dim_customer_normalized = dim_customer \
    .withColumn("MaritalStatus", F.trim(F.upper(F.col("MaritalStatus")))) \
    .withColumn("Gender", F.trim(F.upper(F.col("Gender")))) \
    .withColumn("CustomerName", F.trim(F.col("CustomerName")))

dim_region_normalized = dim_region \
    .withColumn("City", F.trim(F.col("City"))) \
    .withColumn("Region", F.trim(F.col("Region"))) \
    .withColumn("Country", F.trim(F.col("Country")))

dim_product_normalized = dim_product \
    .withColumn("Color", F.trim(F.initcap(F.col("Color")))) \
    .withColumn("Product_Name", F.trim(F.col("Product_Name")))

dim_product_subcategory_normalized = dim_product_subcategory \
    .withColumn("ProductSubcategoryName", F.trim(F.col("ProductSubcategoryName")))

dim_product_category_normalized = dim_product_category \
    .withColumn("ProductCategoryName", F.trim(F.col("ProductCategoryName")))

print("✓ Standardized text columns across all dimension tables")
print("✓ Applied trimming and case normalization")


STEP 4: HANDLING NULLS AND STANDARDIZATION

✓ Filled null Gender values with 'Unknown' in Fact Sales
✓ Standardized text columns across all dimension tables
✓ Applied trimming and case normalization


In [0]:
print("\n" + "="*80)
print("STEP 5: DATA VALIDATION")
print("="*80 + "\n")

# Validate referential integrity
print("Checking referential integrity...\n")

# Check if all ProductKeys in fact table exist in product dimension
fact_products = fact_sales_normalized.select("ProductKey").distinct()
dim_products = dim_product_normalized.select("ProductKey").distinct()
orphan_products = fact_products.join(dim_products, "ProductKey", "left_anti").count()
print(f"Orphan ProductKeys in Fact Sales: {orphan_products}")

# Check if all CustomerKeys in fact table exist in customer dimension
fact_customers = fact_sales_normalized.select("CustomerKey").distinct()
dim_customers = dim_customer_normalized.select("CustomerKey").distinct()
orphan_customers = fact_customers.join(dim_customers, "CustomerKey", "left_anti").count()
print(f"Orphan CustomerKeys in Fact Sales: {orphan_customers}")

# Check if all OrderDateKeys in fact table exist in date dimension
fact_dates = fact_sales_normalized.select("OrderDateKey").distinct()
dim_dates = dim_date.select("DateKey").distinct()
orphan_dates = fact_dates.join(dim_dates.withColumnRenamed("DateKey", "OrderDateKey"), "OrderDateKey", "left_anti").count()
print(f"Orphan OrderDateKeys in Fact Sales: {orphan_dates}")

# Check if all GeographyKeys in customer exist in region dimension
cust_geography = dim_customer_normalized.select("GeographyKey").distinct()
dim_geography = dim_region_normalized.select("GeographyKey").distinct()
orphan_geography = cust_geography.join(dim_geography, "GeographyKey", "left_anti").count()
print(f"Orphan GeographyKeys in Dim Customer: {orphan_geography}")

# Check if all ProductSubcategoryKeys in product exist in subcategory dimension
prod_subcat = dim_product_normalized.select("ProductSubcategoryKey").distinct()
dim_subcat = dim_product_subcategory_normalized.select("ProductSubcategoryKey").distinct()
orphan_subcat = prod_subcat.join(dim_subcat, "ProductSubcategoryKey", "left_anti").count()
print(f"Orphan ProductSubcategoryKeys in Dim Product: {orphan_subcat}")

# Validate data ranges
print("\nValidating data ranges...\n")

# Check for negative values in numeric columns
neg_quantity = fact_sales_normalized.filter(F.col("OrderQuantity") < 0).count()
neg_price = fact_sales_normalized.filter(F.col("List_Price") < 0).count()
neg_cost = fact_sales_normalized.filter(F.col("Product_Cost") < 0).count()

print(f"Negative OrderQuantity: {neg_quantity}")
print(f"Negative List_Price: {neg_price}")
print(f"Negative Product_Cost: {neg_cost}")

if orphan_products == 0 and orphan_customers == 0 and orphan_dates == 0 and orphan_geography == 0 and orphan_subcat == 0:
    print("\n✓ All referential integrity checks passed!")
else:
    print("\n⚠ Some referential integrity issues found - review above")

if neg_quantity == 0 and neg_price == 0 and neg_cost == 0:
    print("✓ All data range validations passed!")
else:
    print("⚠ Some data range issues found - review above")


STEP 5: DATA VALIDATION

Checking referential integrity...

Orphan ProductKeys in Fact Sales: 0
Orphan CustomerKeys in Fact Sales: 7655
Orphan OrderDateKeys in Fact Sales: 0
Orphan GeographyKeys in Dim Customer: 0
Orphan ProductSubcategoryKeys in Dim Product: 0

Validating data ranges...

Negative OrderQuantity: 0
Negative List_Price: 0
Negative Product_Cost: 0

⚠ Some referential integrity issues found - review above
✓ All data range validations passed!


In [0]:
print("\n" + "="*80)
print("STEP 6: JOINING ALL TABLES INTO ONE UNIFIED TABLE")
print("="*80 + "\n")

# Start with fact table
print("Building unified table...")

# Join with customer dimension
unified = fact_sales_normalized.alias("f") \
    .join(
        dim_customer_normalized.alias("c"),
        F.col("f.CustomerKey") == F.col("c.CustomerKey"),
        "left"
    )

print("✓ Joined with Customer dimension")

# Join with region dimension
unified = unified.join(
    dim_region_normalized.alias("r"),
    F.col("c.GeographyKey") == F.col("r.GeographyKey"),
    "left"
)

print("✓ Joined with Region dimension")

# Join with date dimension
unified = unified.join(
    dim_date.alias("d"),
    F.col("f.OrderDateKey") == F.col("d.DateKey"),
    "left"
)

print("✓ Joined with Date dimension")

# Join with product dimension
unified = unified.join(
    dim_product_normalized.alias("p"),
    F.col("f.ProductKey") == F.col("p.ProductKey"),
    "left"
)

print("✓ Joined with Product dimension")

# Join with product subcategory dimension
unified = unified.join(
    dim_product_subcategory_normalized.alias("ps"),
    F.col("p.ProductSubcategoryKey") == F.col("ps.ProductSubcategoryKey"),
    "left"
)

print("✓ Joined with Product Subcategory dimension")

# Join with product category dimension
unified = unified.join(
    dim_product_category_normalized.alias("pc"),
    F.col("ps.ProductCategoryKey") == F.col("pc.ProductCategoryKey"),
    "left"
)

print("✓ Joined with Product Category dimension")

# Select and rename columns for clarity
final_silver_table = unified.select(
    # Order Information
    F.col("f.OrderNumber").alias("order_number"),
    F.col("f.OrderDateKey").alias("order_date_key"),
    F.col("d.FullDateAlternateKey").alias("order_date"),
    F.col("d.CalendarYear").alias("order_year"),
    F.col("d.EnglishMonthName").alias("order_month"),
    
    # Customer Information
    F.col("f.CustomerKey").alias("customer_key"),
    F.col("c.CustomerName").alias("customer_name"),
    F.col("c.Gender").alias("customer_gender"),
    F.col("c.MaritalStatus").alias("customer_marital_status"),
    
    # Geography Information
    F.col("r.GeographyKey").alias("geography_key"),
    F.col("r.City").alias("city"),
    F.col("r.Region").alias("region"),
    F.col("r.Country").alias("country"),
    
    # Product Information
    F.col("f.ProductKey").alias("product_key"),
    F.col("p.Product_Name").alias("product_name"),
    F.col("p.Color").alias("product_color"),
    F.col("ps.ProductSubcategoryName").alias("product_subcategory"),
    F.col("pc.ProductCategoryName").alias("product_category"),
    
    # Sales Metrics
    F.col("f.OrderQuantity").alias("order_quantity"),
    F.col("f.List_Price").alias("list_price"),
    F.col("f.Product_Cost").alias("product_cost"),
    F.col("p.StandardCost").alias("standard_cost"),
    F.col("p.ListPrice").alias("product_list_price"),
    
    # Calculated Metrics
    (F.col("f.OrderQuantity") * F.col("f.List_Price")).alias("total_sales"),
    (F.col("f.OrderQuantity") * F.col("f.Product_Cost")).alias("total_cost"),
    ((F.col("f.OrderQuantity") * F.col("f.List_Price")) - 
     (F.col("f.OrderQuantity") * F.col("f.Product_Cost"))).alias("gross_profit")
)

print("\n✓ Created unified table with all dimensions and calculated metrics")
print(f"\nFinal row count: {final_silver_table.count():,}")
print(f"Total columns: {len(final_silver_table.columns)}")


STEP 6: JOINING ALL TABLES INTO ONE UNIFIED TABLE

Building unified table...
✓ Joined with Customer dimension
✓ Joined with Region dimension
✓ Joined with Date dimension
✓ Joined with Product dimension
✓ Joined with Product Subcategory dimension
✓ Joined with Product Category dimension

✓ Created unified table with all dimensions and calculated metrics

Final row count: 57,188
Total columns: 26


In [0]:
print("\n" + "="*80)
print("SAMPLE OF UNIFIED SILVER TABLE")
print("="*80 + "\n")

# Display schema
print("Schema:")
final_silver_table.printSchema()

print("\nSample data (first 10 rows):")
display(final_silver_table.limit(10))


SAMPLE OF UNIFIED SILVER TABLE

Schema:
root
 |-- order_number: long (nullable = true)
 |-- order_date_key: long (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_year: long (nullable = true)
 |-- order_month: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_marital_status: string (nullable = true)
 |-- geography_key: long (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- product_key: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_color: string (nullable = true)
 |-- product_subcategory: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- order_quantity: long (nullable = true)
 |-- list_price: double (nullable = true)
 |-- product_cost: double (nullable = true)
 |-- standard_cost: double (nullable

order_number,order_date_key,order_date,order_year,order_month,customer_key,customer_name,customer_gender,customer_marital_status,geography_key,city,region,country,product_key,product_name,product_color,product_subcategory,product_category,order_quantity,list_price,product_cost,standard_cost,product_list_price,total_sales,total_cost,gross_profit
20093778,20071207,2007-12-07T00:00:00.000Z,2007,December,22571,Customer3594,M,M,9,Milsons Point,New South Wales,Australia,217,Product 270,Black,Helmets,Accessories,7,34.99,13.0863,13.0863,34.99,244.93,91.6041,153.3259
20093797,20071219,2007-12-19T00:00:00.000Z,2007,December,22578,null,null,null,null,null,null,null,477,Product 155,null,Bottles and Cages,Accessories,19,4.99,1.8663,1.8663,4.99,94.81,35.4597,59.350300000000004
20093783,20071204,2007-12-04T00:00:00.000Z,2007,December,22579,null,null,null,null,null,null,null,225,Product 11,Multi,Caps,Clothing,4,8.99,6.9223,6.9223,8.99,35.96,27.6892,8.270800000000001
20083689,20061107,2006-11-07T00:00:00.000Z,2006,November,22582,null,null,null,null,null,null,null,377,Product 330,Black,Road Bikes,Bikes,7,2181.5625,1320.6838,1320.6838,2181.5625,15270.9375,9244.7866,6026.150900000001
20102696,20080114,2008-01-14T00:00:00.000Z,2008,January,22582,null,null,null,null,null,null,null,214,Product 3,Red,Helmets,Accessories,14,34.99,13.0863,13.0863,34.99,489.86,183.2082,306.6518
20093812,20071228,2007-12-28T00:00:00.000Z,2007,December,22584,null,null,null,null,null,null,null,530,Product 208,null,Tires and Tubes,Accessories,28,4.99,1.8663,1.8663,4.99,139.72,52.2564,87.4636
20093725,20071127,2007-11-27T00:00:00.000Z,2007,November,22598,null,null,null,null,null,null,null,477,Product 155,null,Bottles and Cages,Accessories,27,4.99,1.8663,1.8663,4.99,134.73000000000002,50.390100000000004,84.33990000000001
20093725,20071127,2007-11-27T00:00:00.000Z,2007,November,22598,null,null,null,null,null,null,null,225,Product 11,Multi,Caps,Clothing,27,8.99,6.9223,6.9223,8.99,242.73000000000002,186.9021,55.82790000000003
20103218,20080614,2008-06-14T00:00:00.000Z,2008,June,22604,null,null,null,null,null,null,null,214,Product 3,Red,Helmets,Accessories,14,34.99,13.0863,13.0863,34.99,489.86,183.2082,306.6518
20103219,20080614,2008-06-14T00:00:00.000Z,2008,June,22605,null,null,null,null,null,null,null,581,Product 251,Yellow,Road Bikes,Bikes,14,1700.99,1082.51,1082.51,1700.99,23813.86,15155.14,8658.720000000001


In [0]:
print("\n" + "="*80)
print("STEP 8: SAVING TO SILVER LAYER")
print("="*80 + "\n")

# Save unified table to silver layer
silver_table_name = f"{silver_catalog}.{silver_schema}.fact_sales_unified"

final_silver_table.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table_name)

print(f"✓ Saved unified table to {silver_table_name}")
print(f"✓ Total rows: {spark.table(silver_table_name).count():,}")

# Also save individual normalized dimension tables to silver
print("\nSaving normalized dimension tables to silver...\n")

tables_to_save = [
    (dim_customer_normalized, "dim_customer"),
    (dim_region_normalized, "dim_region"),
    (dim_product_normalized, "dim_product"),
    (dim_product_subcategory_normalized, "dim_product_subcategory"),
    (dim_product_category_normalized, "dim_product_category"),
    (dim_date, "dim_date")
]

for df, table_name in tables_to_save:
    full_table_name = f"{silver_catalog}.{silver_schema}.{table_name}"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)
    print(f"✓ Saved {full_table_name} ({df.count():,} rows)")

print("\n" + "="*80)
print("SILVER LAYER CREATION COMPLETE!")
print("="*80)


STEP 8: SAVING TO SILVER LAYER

✓ Saved unified table to workspace.silver.fact_sales_unified
✓ Total rows: 57,188

Saving normalized dimension tables to silver...

✓ Saved workspace.silver.dim_customer (9,999 rows)
✓ Saved workspace.silver.dim_region (655 rows)
✓ Saved workspace.silver.dim_product (400 rows)
✓ Saved workspace.silver.dim_product_subcategory (40 rows)
✓ Saved workspace.silver.dim_product_category (5 rows)
✓ Saved workspace.silver.dim_date (1,188 rows)

SILVER LAYER CREATION COMPLETE!


In [0]:
print("\n" + "="*80)
print("APPLYING BUSINESS LOGIC")
print("="*80 + "\n")

# Load the unified table
df = spark.table(f"{silver_catalog}.{silver_schema}.fact_sales_unified")

print("Adding business metrics and calculated fields...\n")

# 1. Profitability Metrics
df_enriched = df.withColumn(
    "profit_margin_pct",
    F.round(
        F.when(F.col("total_sales") > 0, 
               (F.col("gross_profit") / F.col("total_sales")) * 100
        ).otherwise(0),
        2
    )
).withColumn(
    "markup_pct",
    F.round(
        F.when(F.col("total_cost") > 0,
               ((F.col("total_sales") - F.col("total_cost")) / F.col("total_cost")) * 100
        ).otherwise(0),
        2
    )
)

print("✓ Added profit margin % and markup %")

# 2. Order Value Categorization
df_enriched = df_enriched.withColumn(
    "order_value_category",
    F.when(F.col("total_sales") >= 5000, "High Value")
    .when(F.col("total_sales") >= 1000, "Medium Value")
    .when(F.col("total_sales") >= 100, "Low Value")
    .otherwise("Micro Transaction")
)

print("✓ Categorized order values (High/Medium/Low/Micro)")

# 3. Profitability Classification
df_enriched = df_enriched.withColumn(
    "profitability_status",
    F.when(F.col("profit_margin_pct") >= 40, "Highly Profitable")
    .when(F.col("profit_margin_pct") >= 25, "Profitable")
    .when(F.col("profit_margin_pct") >= 10, "Moderately Profitable")
    .when(F.col("profit_margin_pct") > 0, "Low Profit")
    .otherwise("Unprofitable")
)

print("✓ Classified profitability status")

# 4. Product Price Tier
df_enriched = df_enriched.withColumn(
    "price_tier",
    F.when(F.col("list_price") >= 1000, "Premium")
    .when(F.col("list_price") >= 100, "Mid-Range")
    .otherwise("Budget")
)

print("✓ Added product price tiers")

# 5. Bulk Order Flag
df_enriched = df_enriched.withColumn(
    "is_bulk_order",
    F.when(F.col("order_quantity") >= 20, True).otherwise(False)
)

print("✓ Flagged bulk orders (quantity >= 20)")


APPLYING BUSINESS LOGIC

Adding business metrics and calculated fields...

✓ Added profit margin % and markup %
✓ Categorized order values (High/Medium/Low/Micro)
✓ Classified profitability status
✓ Added product price tiers
✓ Flagged bulk orders (quantity >= 20)


In [0]:
print("\n" + "="*80)
print("CUSTOMER SEGMENTATION - RFM ANALYSIS")
print("="*80 + "\n")

# Calculate customer-level metrics
customer_metrics = df_enriched.groupBy("customer_key").agg(
    F.count("order_number").alias("total_orders"),
    F.sum("total_sales").alias("lifetime_value"),
    F.sum("order_quantity").alias("total_items_purchased"),
    F.avg("total_sales").alias("avg_order_value"),
    F.max("order_date").alias("last_purchase_date"),
    F.min("order_date").alias("first_purchase_date")
)

# Calculate days since last purchase (recency)
from datetime import datetime
current_date = F.lit(datetime(2008, 12, 31))  # Using max date in dataset

customer_metrics = customer_metrics.withColumn(
    "days_since_last_purchase",
    F.datediff(current_date, F.col("last_purchase_date"))
).withColumn(
    "customer_tenure_days",
    F.datediff(F.col("last_purchase_date"), F.col("first_purchase_date"))
)

print("✓ Calculated customer-level metrics")

# RFM Scoring (1-5, where 5 is best)
# Recency Score (lower days = higher score)
customer_metrics = customer_metrics.withColumn(
    "recency_score",
    F.when(F.col("days_since_last_purchase") <= 30, 5)
    .when(F.col("days_since_last_purchase") <= 90, 4)
    .when(F.col("days_since_last_purchase") <= 180, 3)
    .when(F.col("days_since_last_purchase") <= 365, 2)
    .otherwise(1)
)

# Frequency Score (more orders = higher score)
customer_metrics = customer_metrics.withColumn(
    "frequency_score",
    F.when(F.col("total_orders") >= 20, 5)
    .when(F.col("total_orders") >= 10, 4)
    .when(F.col("total_orders") >= 5, 3)
    .when(F.col("total_orders") >= 2, 2)
    .otherwise(1)
)

# Monetary Score (higher value = higher score)
customer_metrics = customer_metrics.withColumn(
    "monetary_score",
    F.when(F.col("lifetime_value") >= 10000, 5)
    .when(F.col("lifetime_value") >= 5000, 4)
    .when(F.col("lifetime_value") >= 1000, 3)
    .when(F.col("lifetime_value") >= 100, 2)
    .otherwise(1)
)

# Overall RFM Score
customer_metrics = customer_metrics.withColumn(
    "rfm_score",
    F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score")
)

print("✓ Calculated RFM scores (Recency, Frequency, Monetary)")

# Customer Segment Classification
customer_metrics = customer_metrics.withColumn(
    "customer_segment",
    F.when(F.col("rfm_score") >= 13, "Champions")
    .when((F.col("rfm_score") >= 10) & (F.col("recency_score") >= 4), "Loyal Customers")
    .when((F.col("rfm_score") >= 10) & (F.col("recency_score") < 4), "At Risk")
    .when((F.col("rfm_score") >= 7) & (F.col("monetary_score") >= 4), "Big Spenders")
    .when((F.col("rfm_score") >= 7), "Potential Loyalists")
    .when((F.col("recency_score") >= 4) & (F.col("frequency_score") <= 2), "New Customers")
    .when((F.col("recency_score") <= 2) & (F.col("frequency_score") >= 3), "Hibernating")
    .when(F.col("recency_score") <= 2, "Lost Customers")
    .otherwise("Occasional Buyers")
)

print("✓ Segmented customers into 9 categories")

# Display segment distribution
print("\nCustomer Segment Distribution:")
segment_dist = customer_metrics.groupBy("customer_segment").count().orderBy(F.desc("count"))
display(segment_dist)


CUSTOMER SEGMENTATION - RFM ANALYSIS

✓ Calculated customer-level metrics
✓ Calculated RFM scores (Recency, Frequency, Monetary)
✓ Segmented customers into 9 categories

Customer Segment Distribution:


customer_segment,count
Big Spenders,6298
Lost Customers,6088
Potential Loyalists,2849
At Risk,1644
Occasional Buyers,95
Hibernating,29
Champions,25


In [0]:
print("\n" + "="*80)
print("PRODUCT PERFORMANCE & ABC ANALYSIS")
print("="*80 + "\n")

# Calculate product-level metrics
product_metrics = df_enriched.groupBy(
    "product_key", "product_name", "product_category", "product_subcategory"
).agg(
    F.sum("total_sales").alias("total_revenue"),
    F.sum("gross_profit").alias("total_profit"),
    F.sum("order_quantity").alias("units_sold"),
    F.count("order_number").alias("number_of_orders"),
    F.avg("profit_margin_pct").alias("avg_profit_margin_pct")
)

print("✓ Calculated product-level performance metrics")

# Calculate cumulative percentage for ABC analysis
window_spec = Window.orderBy(F.desc("total_revenue"))

product_metrics = product_metrics.withColumn(
    "revenue_rank",
    F.row_number().over(window_spec)
).withColumn(
    "cumulative_revenue",
    F.sum("total_revenue").over(window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow))
)

# Calculate total revenue for percentage
total_revenue = product_metrics.agg(F.sum("total_revenue")).collect()[0][0]

product_metrics = product_metrics.withColumn(
    "revenue_contribution_pct",
    F.round((F.col("total_revenue") / F.lit(total_revenue)) * 100, 2)
).withColumn(
    "cumulative_revenue_pct",
    F.round((F.col("cumulative_revenue") / F.lit(total_revenue)) * 100, 2)
)

print("✓ Calculated revenue contribution percentages")

# ABC Classification
# A: Top 20% of revenue (typically 80% of sales)
# B: Next 30% of revenue (typically 15% of sales)
# C: Remaining 50% (typically 5% of sales)
product_metrics = product_metrics.withColumn(
    "abc_classification",
    F.when(F.col("cumulative_revenue_pct") <= 80, "A - High Value")
    .when(F.col("cumulative_revenue_pct") <= 95, "B - Medium Value")
    .otherwise("C - Low Value")
)

print("✓ Applied ABC classification")

# Product Performance Status
product_metrics = product_metrics.withColumn(
    "performance_status",
    F.when((F.col("abc_classification") == "A - High Value") & (F.col("avg_profit_margin_pct") >= 30), "Star Product")
    .when((F.col("abc_classification") == "A - High Value"), "High Volume")
    .when((F.col("abc_classification") == "C - Low Value") & (F.col("units_sold") < 100), "Slow Mover")
    .when(F.col("avg_profit_margin_pct") < 10, "Low Margin")
    .otherwise("Standard")
)

print("✓ Classified product performance status")

# Display ABC distribution
print("\nABC Classification Distribution:")
abc_dist = product_metrics.groupBy("abc_classification").agg(
    F.count("*").alias("product_count"),
    F.sum("total_revenue").alias("total_revenue"),
    F.round(F.sum("revenue_contribution_pct"), 2).alias("revenue_pct")
).orderBy("abc_classification")
display(abc_dist)

print("\nTop 10 Products by Revenue:")
display(product_metrics.orderBy(F.desc("total_revenue")).limit(10))


PRODUCT PERFORMANCE & ABC ANALYSIS

✓ Calculated product-level performance metrics


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Calculated revenue contribution percentages
✓ Applied ABC classification
✓ Classified product performance status

ABC Classification Distribution:


abc_classification,product_count,total_revenue,revenue_pct
A - High Value,46,3.4392624565709996E8,79.85
B - Medium Value,44,6.545517534749998E7,15.19
C - Low Value,68,2.1545886893000003E7,5.02



Top 10 Products by Revenue:


product_key,product_name,product_category,product_subcategory,total_revenue,total_profit,units_sold,number_of_orders,avg_profit_margin_pct,revenue_rank,cumulative_revenue,revenue_contribution_pct,cumulative_revenue_pct,abc_classification,performance_status
359,Product 317,Bikes,Mountain Bikes,1.6025915170000013E7,7283329.752099999,6983,416,45.45000000000035,1,1.6025915170000013E7,3.72,3.72,A - High Value,Star Product
310,Product 67,Bikes,Road Bikes,1.5844579559999993E7,6230088.8423999995,4428,267,39.31999999999985,2,3.1870494730000004E7,3.68,7.4,A - High Value,Star Product
353,Product 94,Bikes,Mountain Bikes,1.569009237E7,7130707.691499999,6763,422,45.45000000000036,3,4.75605871E7,3.64,11.04,A - High Value,Star Product
363,Product 321,Bikes,Mountain Bikes,1.548659252000001E7,7038222.707599996,6748,419,45.45000000000035,4,6.304717962000001E7,3.59,14.63,A - High Value,Star Product
361,Product 319,Bikes,Mountain Bikes,1.5119394120000005E7,6871341.315599998,6588,427,45.45000000000036,5,7.816657374000001E7,3.51,18.14,A - High Value,Star Product
355,Product 96,Bikes,Mountain Bikes,1.4546337300000016E7,6610903.035,6270,391,45.45000000000033,6,9.271291104000002E7,3.38,21.51,A - High Value,Star Product
312,Product 69,Bikes,Road Bikes,1.4427584640000008E7,5672926.425600001,4032,266,39.31999999999985,7,1.0714049568000004E8,3.35,24.86,A - High Value,Star Product
357,Product 98,Bikes,Mountain Bikes,1.3964019810000017E7,6346256.039500006,6019,401,45.45000000000034,8,1.2110451549000005E8,3.24,28.1,A - High Value,Star Product
313,Product 70,Bikes,Road Bikes,1.395525299999999E7,5487205.62,3900,244,39.319999999999865,9,1.3505976849000004E8,3.24,31.34,A - High Value,Star Product
314,Product 71,Bikes,Road Bikes,1.3937361649999987E7,5480170.740999995,3895,238,39.31999999999987,10,1.4899713014000002E8,3.23,34.58,A - High Value,Star Product


In [0]:
print("\n" + "="*80)
print("TIME-BASED ANALYSIS & SEASONALITY")
print("="*80 + "\n")

# Add time-based features
df_time = df_enriched.withColumn(
    "quarter",
    F.quarter(F.col("order_date"))
).withColumn(
    "month_num",
    F.month(F.col("order_date"))
).withColumn(
    "day_of_week",
    F.dayofweek(F.col("order_date"))
).withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False)
)

print("✓ Added quarter, month, day of week features")

# Season classification
df_time = df_time.withColumn(
    "season",
    F.when(F.col("month_num").isin([12, 1, 2]), "Winter")
    .when(F.col("month_num").isin([3, 4, 5]), "Spring")
    .when(F.col("month_num").isin([6, 7, 8]), "Summer")
    .otherwise("Fall")
)

print("✓ Classified seasons")

# Calculate year-over-year growth by product category
print("\nCalculating year-over-year growth...")

yoy_growth = df_time.groupBy("product_category", "order_year").agg(
    F.sum("total_sales").alias("yearly_sales")
)

window_yoy = Window.partitionBy("product_category").orderBy("order_year")

yoy_growth = yoy_growth.withColumn(
    "previous_year_sales",
    F.lag("yearly_sales", 1).over(window_yoy)
).withColumn(
    "yoy_growth_pct",
    F.round(
        F.when(F.col("previous_year_sales").isNotNull(),
               ((F.col("yearly_sales") - F.col("previous_year_sales")) / F.col("previous_year_sales")) * 100
        ).otherwise(None),
        2
    )
)

print("✓ Calculated year-over-year growth by category")

print("\nYear-over-Year Growth by Category:")
display(yoy_growth.orderBy("product_category", "order_year"))

# Seasonal performance by category
print("\nSeasonal Performance Analysis:")
seasonal_sales = df_time.groupBy("product_category", "season").agg(
    F.sum("total_sales").alias("season_sales"),
    F.count("order_number").alias("order_count")
).orderBy("product_category", "season")

display(seasonal_sales)


TIME-BASED ANALYSIS & SEASONALITY

✓ Added quarter, month, day of week features
✓ Classified seasons

Calculating year-over-year growth...
✓ Calculated year-over-year growth by category

Year-over-Year Growth by Category:


product_category,order_year,yearly_sales,previous_year_sales,yoy_growth_pct
Accessories,2007,4423332.810000086,null,null
Accessories,2008,6140025.650000173,4423332.810000086,38.81
Bikes,2005,3.85286885094E7,null,null
Bikes,2006,9.564757992989978E7,3.85286885094E7,148.25
Bikes,2007,1.4158488805829957E8,9.564757992989978E7,48.03
Bikes,2008,1.3959601854999942E8,1.4158488805829957E8,-1.4
Clothing,2007,2036801.3299999838,null,null
Clothing,2008,2969973.0600000173,2036801.3299999838,45.82



Seasonal Performance Analysis:


product_category,season,season_sales,order_count
Accessories,Fall,2389514.2499999846,7819
Accessories,Spring,2971511.780000009,9406
Accessories,Summer,2528239.359999999,8099
Accessories,Winter,2674093.0700000054,8959
Bikes,Fall,7.82349969036999E7,2728
Bikes,Spring,1.2817396570029952E8,4378
Bikes,Summer,1.0242358228239952E8,3382
Bikes,Winter,1.0652463016119972E8,3818
Clothing,Fall,1113666.0199999954,1906
Clothing,Spring,1372018.15999999,2333


In [0]:
print("\n" + "="*80)
print("SAVING FINAL ENRICHED TABLE TO SILVER LAYER")
print("="*80 + "\n")

# Join time-based features with customer segments and product classifications
print("Joining all business logic together...\n")

# Join with customer metrics
gold_table = df_time.join(
    customer_metrics.select(
        "customer_key",
        "total_orders",
        "lifetime_value",
        "avg_order_value",
        "days_since_last_purchase",
        "customer_tenure_days",
        "rfm_score",
        "customer_segment"
    ),
    "customer_key",
    "left"
)

print("✓ Joined with customer RFM segmentation")

# Join with product metrics
gold_table = gold_table.join(
    product_metrics.select(
        "product_key",
        "abc_classification",
        "performance_status",
        "revenue_contribution_pct"
    ),
    "product_key",
    "left"
)

print("✓ Joined with product ABC classification")

# Add business rules flags
gold_table = gold_table.withColumn(
    "needs_attention",
    F.when(
        (F.col("customer_segment").isin(["At Risk", "Hibernating", "Lost Customers"])) |
        (F.col("performance_status") == "Slow Mover") |
        (F.col("profitability_status") == "Unprofitable"),
        True
    ).otherwise(False)
).withColumn(
    "high_value_transaction",
    F.when(
        (F.col("order_value_category") == "High Value") &
        (F.col("customer_segment").isin(["Champions", "Loyal Customers", "Big Spenders"])),
        True
    ).otherwise(False)
)

print("✓ Added business rule flags (needs_attention, high_value_transaction)")

# Save to Silver layer as the final enriched table
print("\nSaving to Silver layer...")

silver_catalog = "workspace"
silver_schema = "silver"

# Silver schema already exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")

silver_table_name = f"{silver_catalog}.{silver_schema}.fact_sales_enriched"

gold_table.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table_name)

print(f"\n✓ Saved final enriched table to {silver_table_name}")
print(f"✓ Total rows: {spark.table(silver_table_name).count():,}")
print(f"✓ Total columns: {len(gold_table.columns)}")

print("\n" + "="*80)
print("SILVER LAYER WITH BUSINESS LOGIC COMPLETE!")
print("="*80)


SAVING FINAL ENRICHED TABLE TO SILVER LAYER

Joining all business logic together...

✓ Joined with customer RFM segmentation
✓ Joined with product ABC classification
✓ Added business rule flags (needs_attention, high_value_transaction)

Saving to Silver layer...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



✓ Saved final enriched table to workspace.silver.fact_sales_enriched
✓ Total rows: 57,188
✓ Total columns: 49

SILVER LAYER WITH BUSINESS LOGIC COMPLETE!


In [0]:
print("\n" + "="*80)
print("BUSINESS INSIGHTS SUMMARY")
print("="*80 + "\n")

# Load the enriched silver table
silver_catalog = "workspace"
silver_schema = "silver"
enriched_df = spark.table(f"{silver_catalog}.{silver_schema}.fact_sales_enriched")

# Key Business Metrics
print("KEY BUSINESS METRICS")
print("-" * 80)

total_revenue = enriched_df.agg(F.sum("total_sales")).collect()[0][0]
total_profit = enriched_df.agg(F.sum("gross_profit")).collect()[0][0]
avg_margin = enriched_df.agg(F.avg("profit_margin_pct")).collect()[0][0]
total_customers = enriched_df.select("customer_key").distinct().count()
total_orders = enriched_df.count()
avg_order_value = total_revenue / total_orders if total_orders > 0 else 0

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Overall Profit Margin: {avg_margin:.2f}%")
print(f"Total Customers: {total_customers:,}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: ${avg_order_value:,.2f}")

# Customer Segment Performance
print("\n\nCUSTOMER SEGMENT PERFORMANCE")
print("-" * 80)

customer_segment_perf = enriched_df.groupBy("customer_segment").agg(
    F.countDistinct("customer_key").alias("customer_count"),
    F.sum("total_sales").alias("segment_revenue"),
    F.avg("lifetime_value").alias("avg_lifetime_value"),
    F.avg("rfm_score").alias("avg_rfm_score")
).withColumn(
    "revenue_pct",
    F.round((F.col("segment_revenue") / F.lit(total_revenue)) * 100, 2)
).orderBy(F.desc("segment_revenue"))

display(customer_segment_perf)

# Product Category Performance
print("\nPRODUCT CATEGORY PERFORMANCE")
print("-" * 80)

category_perf = enriched_df.groupBy("product_category").agg(
    F.sum("total_sales").alias("category_revenue"),
    F.sum("gross_profit").alias("category_profit"),
    F.avg("profit_margin_pct").alias("avg_margin_pct"),
    F.sum("order_quantity").alias("units_sold")
).withColumn(
    "revenue_pct",
    F.round((F.col("category_revenue") / F.lit(total_revenue)) * 100, 2)
).orderBy(F.desc("category_revenue"))

display(category_perf)

# Geographic Performance
print("\nTOP 10 CITIES BY REVENUE")
print("-" * 80)

city_perf = enriched_df.groupBy("city", "region", "country").agg(
    F.sum("total_sales").alias("city_revenue"),
    F.count("order_number").alias("order_count")
).orderBy(F.desc("city_revenue")).limit(10)

display(city_perf)

# High Priority Actions
print("\nHIGH PRIORITY ACTIONS")
print("-" * 80)

at_risk_customers = enriched_df.filter(
    F.col("customer_segment").isin(["At Risk", "Hibernating", "Lost Customers"])
).select("customer_key").distinct().count()

slow_moving_products = enriched_df.filter(
    F.col("performance_status") == "Slow Mover"
).select("product_key").distinct().count()

unprofitable_orders = enriched_df.filter(
    F.col("profitability_status") == "Unprofitable"
).count()

print(f"At-Risk/Hibernating/Lost Customers: {at_risk_customers:,} (Needs retention campaign)")
print(f"Slow Moving Products: {slow_moving_products:,} (Consider clearance or discontinuation)")
print(f"Unprofitable Orders: {unprofitable_orders:,} (Review pricing strategy)")

# Top Opportunities
print("\n\nTOP OPPORTUNITIES")
print("-" * 80)

champions = enriched_df.filter(F.col("customer_segment") == "Champions").select("customer_key").distinct().count()
star_products = enriched_df.filter(F.col("performance_status") == "Star Product").select("product_key").distinct().count()
high_value_trans = enriched_df.filter(F.col("high_value_transaction") == True).count()

print(f"Champion Customers: {champions:,} (Focus on retention and upselling)")
print(f"Star Products: {star_products:,} (Invest in inventory and marketing)")
print(f"High-Value Transactions: {high_value_trans:,} (Analyze and replicate success patterns)")


BUSINESS INSIGHTS SUMMARY

KEY BUSINESS METRICS
--------------------------------------------------------------------------------
Total Revenue: $430,927,307.90
Total Profit: $177,719,486.39
Overall Profit Margin: 53.41%
Total Customers: 17,028
Total Orders: 57,188
Average Order Value: $7,535.28


CUSTOMER SEGMENT PERFORMANCE
--------------------------------------------------------------------------------


customer_segment,customer_count,segment_revenue,avg_lifetime_value,avg_rfm_score,revenue_pct
Big Spenders,6298,2.9213707158510005E8,50071.1542446049,8.57603501182152,67.79
At Risk,1644,1.2837458175690006E8,79267.13047775847,10.158330268481059,29.79
Potential Loyalists,2849,5439438.131699996,2064.486277600835,7.596503375454388,1.26
Lost Customers,6088,4526771.453899994,789.2078771679361,5.473598096088056,1.05
Champions,25,405378.14999999985,16888.09288020391,13.0,0.09
Occasional Buyers,95,26240.6,258.5265384615385,5.769230769230769,0.01
Hibernating,29,17826.219999999998,615.417697368421,6.0,0.0



PRODUCT CATEGORY PERFORMANCE
--------------------------------------------------------------------------------


product_category,category_revenue,category_profit,avg_margin_pct,units_sold,revenue_pct
Bikes,4.1535717504760104E8,1.6909824219539812E8,40.33437718439968,225585,96.39
Accessories,1.056335846000056E7,6612645.124599967,62.6000000000396,539460,2.45
Clothing,5006774.390000059,2008599.0677000158,38.54250494243472,134540,1.16



TOP 10 CITIES BY REVENUE
--------------------------------------------------------------------------------


city,region,country,city_revenue,order_count
null,null,null,2.0577159365859726E8,25325
London,England,United Kingdom,7529147.946599997,853
Paris,Seine (Paris),France,4437925.117200007,548
Bendigo,Victoria,Australia,3091954.334999998,227
Wollongong,New South Wales,Australia,2889235.3962000003,226
Warrnambool,Victoria,Australia,2588512.3433,214
Bellflower,California,United States,2551116.9323999994,285
Brisbane,Queensland,Australia,2307814.008399999,201
Sydney,New South Wales,Australia,2301753.4986000014,205
Goulburn,New South Wales,Australia,2160694.480599999,153



HIGH PRIORITY ACTIONS
--------------------------------------------------------------------------------
At-Risk/Hibernating/Lost Customers: 7,761 (Needs retention campaign)
Slow Moving Products: 0 (Consider clearance or discontinuation)
Unprofitable Orders: 0 (Review pricing strategy)


TOP OPPORTUNITIES
--------------------------------------------------------------------------------
Champion Customers: 25 (Focus on retention and upselling)
Star Products: 46 (Invest in inventory and marketing)
High-Value Transactions: 9,063 (Analyze and replicate success patterns)
